# Step 1.2b – FinBERT + BERTopic NLP Pipeline for US–China Sentiment

**Thesis: Geopolitical Turning Points and Macroeconomic Volatility — Extension of Saadaoui (2026)**

---

## Notebook Overview

This notebook fulfills the supervisor's plan requirements for:
1. **FinBERT sentiment analysis** (Zhang 2025 methodology) — pre-trained financial NLP on news headlines
2. **BERTopic topic modeling** — unsupervised topic extraction for geopolitical theme identification  
3. **Monthly aggregation** — collapse to time-series features for the feature matrix

### Data Strategy: Cache-First

| Data Source | Cache File | Status |
|-------------|-----------|--------|
| GDELT URLs (2015-2022) | `uschn_urls_2015_2022.parquet` | Load if exists |
| FinBERT article scores | `finbert_article_level.csv` | Load if exists |
| FinBERT monthly | `finbert_sentiment_monthly.csv` | Load if exists |
| BERTopic input texts | `bertopic_input.csv` | Load if exists |
| BERTopic monthly proportions | `bertopic_topics_monthly.csv` | Load if exists |
| BERTopic topic info | `bertopic_topic_info.csv` | Load if exists |

**Rule: If cache exists → reload. If missing → recompute. No unnecessary downloads.**

### Coverage & Known Issues

| Period | Status | Notes |
|--------|--------|-------|
| 2015-04 to 2016-12 | ✅ Full | Good URL extraction |
| 2017-01 to 2017-07 | ✅ Full | Normal operation |
| **2017-08 to 2017-12** | ❌ **All zeros** | GDELT API errors — **will retry** |
| 2018-01 to 2022-02 | ✅ Full | Normal operation |

**Action:** Re-fetch 2017-08 through 2017-12 using retry logic with exponential backoff.


In [1]:
import pandas as pd
import numpy as np
import time
import gc
import warnings
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import random

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / 'data').exists() else NOTEBOOK_DIR
DATA_DIR     = PROJECT_ROOT / 'data'
NLP_DIR      = DATA_DIR / '03_nlp'
NLP_DIR.mkdir(parents=True, exist_ok=True)

# ── Sample window ─────────────────────────────────────────────────────────────
START = '2015-04-01'
END   = '2022-02-28'

print("=" * 70)
print("SETUP COMPLETE")
print(f"Project root: {PROJECT_ROOT}")
print(f"NLP data dir: {NLP_DIR}")
print(f"Date range: {START} to {END}")
print("=" * 70)

# Check what cache files exist
cache_files = [
    'uschn_urls_2015_2022.parquet',
    'finbert_article_level.csv', 
    'finbert_sentiment_monthly.csv',
    'bertopic_input.csv',
    'bertopic_topics_monthly.csv',
    'bertopic_topic_info.csv'
]

print("Cache file status:")
for f in cache_files:
    path = NLP_DIR / f
    status = "EXISTS" if path.exists() else "MISSING"
    size = f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else ""
    print(f"  {f:40s} {status:8s} {size}")


SETUP COMPLETE
Project root: C:\Users\HP\Desktop\replication+contribution
NLP data dir: C:\Users\HP\Desktop\replication+contribution\data\03_nlp
Date range: 2015-04-01 to 2022-02-28
Cache file status:
  uschn_urls_2015_2022.parquet             EXISTS   (23480.0 KB)
  finbert_article_level.csv                EXISTS   (1842.6 KB)
  finbert_sentiment_monthly.csv            EXISTS   (6.3 KB)
  bertopic_input.csv                       EXISTS   (9721.3 KB)
  bertopic_topics_monthly.csv              EXISTS   (15.5 KB)
  bertopic_topic_info.csv                  EXISTS   (984.1 KB)


## Part 1: GDELT URL Collection

### Strategy
1. **Load from cache** if `uschn_urls_2015_2022.parquet` exists
2. **Identify 2017 gaps** — months with 0 events (API errors)
3. **Retry 2017** with exponential backoff and longer timeouts
4. **Merge and save** updated cache

### Why 2017 Failed
GDELT 1.0 API experienced intermittent outages in mid-late 2017. The original
fetch returned `None: None` errors for August–December 2017. We retry with:
- Longer timeout (30s vs 20s)
- Multiple retry attempts (3x per month)
- Exponential backoff between retries


In [2]:
# ── GDELT URL Fetching ────────────────────────────────────────────────────────
from gdelt import gdelt

gd = gdelt(version=1)
US_CODE = "USA"
CHN_CODE = "CHN"

def get_uschn_events_with_url(year, month, timeout=30, max_retries=3):
    """Fetch events for one month with retry logic."""
    start = f"{year}-{month:02d}-01"
    if month == 12:
        end = f"{year}-12-31"
    else:
        next_month = month + 1
        end = f"{year}-{next_month:02d}-01"
        end = pd.Timestamp(end) - pd.Timedelta(days=1)
        end = end.strftime("%Y-%m-%d")

    for attempt in range(max_retries):
        try:
            df = gd.Search([start, end], table='events', output='pd', coverage=True)
            if df is None or df.empty:
                return pd.DataFrame()
            needed = ['SQLDATE','Actor1Code','Actor2Code','SOURCEURL']
            if not all(c in df.columns for c in needed):
                return pd.DataFrame()
            mask = ((df['Actor1Code'] == US_CODE) & (df['Actor2Code'] == CHN_CODE)) |                    ((df['Actor1Code'] == CHN_CODE) & (df['Actor2Code'] == US_CODE))
            df = df.loc[mask, needed]
            df['date'] = pd.to_datetime(df['SQLDATE'], format='%Y%m%d')
            df = df[df['SOURCEURL'].notna() & (df['SOURCEURL'] != '')]
            return df[['date', 'SOURCEURL']]
        except Exception as e:
            print(f"  GDELT error {year}-{month:02d} (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                return pd.DataFrame()

# ── Load or build URL cache ──────────────────────────────────────────────────
cache_path = NLP_DIR / 'uschn_urls_2015_2022.parquet'

if cache_path.exists():
    df_urls = pd.read_parquet(cache_path)
    print(f"Loaded {len(df_urls):,} events with URLs from cache.")

    # Check for 2017 gaps
    df_urls['year_month'] = df_urls['date'].dt.to_period('M')
    monthly_counts = df_urls.groupby('year_month').size()

    print("Monthly event counts (2017):")
    for ym in pd.period_range('2017-01', '2017-12', freq='M'):
        count = monthly_counts.get(ym, 0)
        status = "ZERO" if count == 0 else "OK"
        print(f"  {ym}: {count:5d} events [{status}]")

    zero_months = [(ym.year, ym.month) for ym in pd.period_range('2017-01', '2017-12', freq='M') 
                   if monthly_counts.get(ym, 0) == 0]

    if zero_months:
        print(f"[!] Found {len(zero_months)} months with zero events. Retrying...")

        retry_frames = []
        for year, month in zero_months:
            print(f"Retrying {year}-{month:02d}...")
            df_month = get_uschn_events_with_url(year, month, timeout=30, max_retries=3)
            if not df_month.empty:
                print(f"  [OK] Recovered {len(df_month)} events!")
                retry_frames.append(df_month)
            else:
                print(f"  [FAIL] Still failed after retries.")
            time.sleep(1)

        if retry_frames:
            df_retry = pd.concat(retry_frames, ignore_index=True)
            df_urls = pd.concat([df_urls, df_retry], ignore_index=True)
            df_urls = df_urls.drop_duplicates(subset=['date', 'SOURCEURL'])
            df_urls.to_parquet(cache_path)
            print(f"[OK] Cache updated! Added {len(df_retry)} events. Total: {len(df_urls):,}")
        else:
            print("[!] No additional events recovered. Using original cache.")
    else:
        print("[OK] No gaps found in 2017. Cache is complete.")

else:
    print("No cache found. Building from scratch (this will take time)...")
    all_urls = []
    for yr in range(2015, 2023):
        for mo in range(1, 13):
            if yr == 2015 and mo < 4: continue
            if yr == 2022 and mo > 2: continue
            df_month = get_uschn_events_with_url(yr, mo)
            all_urls.append(df_month)
            print(f"  {yr}-{mo:02d}: {len(df_month)} events with URL")
            gc.collect()
    df_urls = pd.concat(all_urls, ignore_index=True)
    df_urls.to_parquet(cache_path)
    print(f"Saved {len(df_urls):,} events to {cache_path}")

print(f"Final dataset: {len(df_urls):,} events")
print(f"Date range: {df_urls['date'].min()} to {df_urls['date'].max()}")


here
Loaded 387,073 events with URLs from cache.
Monthly event counts (2017):
  2017-01:  6224 events [OK]
  2017-02:  5142 events [OK]
  2017-03:  5503 events [OK]
  2017-04:  7140 events [OK]
  2017-05:  5310 events [OK]
  2017-06:  5271 events [OK]
  2017-07:  5068 events [OK]
  2017-08:   192 events [OK]
  2017-09:  4201 events [OK]
  2017-10:   109 events [OK]
  2017-11:    77 events [OK]
  2017-12:   180 events [OK]
[OK] No gaps found in 2017. Cache is complete.
Final dataset: 387,073 events
Date range: 1920-01-01 00:00:00 to 2022-02-28 00:00:00


## Part 2: Sampling Strategy

### Methodology (Zhang 2025)
- Randomly sample **N = 2,000 articles per year** for the US–China dyad
- This balances coverage across years while keeping computation tractable
- Sample is stratified by year (not month) to ensure each year is represented

### Cache Check
If `bertopic_input.csv` exists, we skip sampling and text extraction entirely.


In [3]:
# ── Sampling ──────────────────────────────────────────────────────────────────
BERTOPIC_INPUT = NLP_DIR / 'bertopic_input.csv'

if BERTOPIC_INPUT.exists():
    print("[OK] bertopic_input.csv found. Loading from cache...")
    df_bertopic_input = pd.read_csv(BERTOPIC_INPUT)
    print(f"Loaded {len(df_bertopic_input):,} articles with extracted text.")
    print(f"Date range: {df_bertopic_input['date'].min()} to {df_bertopic_input['date'].max()}")
else:
    print("[!] bertopic_input.csv not found. Running sampling + text extraction...")

    # Year-stratified sampling: 2,000 per year
    random.seed(42)
    df_urls['year'] = df_urls['date'].dt.year
    sample_frames = []

    for yr in range(2015, 2023):
        yr_df = df_urls[df_urls['year'] == yr]
        n_sample = min(2000, len(yr_df))
        if n_sample > 0:
            sample = yr_df.sample(n=n_sample, random_state=42)
            sample_frames.append(sample)
            print(f"  {yr}: sampled {n_sample:,} / {len(yr_df):,} available")

    df_sample = pd.concat(sample_frames, ignore_index=True)
    print(f"Total sampled: {len(df_sample):,} articles")

    # Text extraction with newspaper3k
    from newspaper import Article

    def extract_text_from_url(url):
        """Extract headline + first 1000 chars from URL."""
        try:
            article = Article(url, language='en')
            article.download()
            article.parse()
            title = article.title or ''
            text = article.text or ''
            if len(text) < 50:
                return None
            return title + '. ' + text[:1000]
        except Exception:
            return None

    results = []
    for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Extracting text"):
        url = row['SOURCEURL']
        text = extract_text_from_url(url)
        if text:
            results.append({'date': row['date'], 'text': text, 'url': url})
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx+1:,} / {len(df_sample):,}...")

    df_bertopic_input = pd.DataFrame(results)
    df_bertopic_input.to_csv(BERTOPIC_INPUT, index=False)
    print(f"[OK] Saved {len(df_bertopic_input):,} articles to {BERTOPIC_INPUT}")

print(f"Final BERTopic input: {len(df_bertopic_input):,} articles")
print(df_bertopic_input.head(3).to_string())


[OK] bertopic_input.csv found. Loading from cache...
Loaded 2,079 articles with extracted text.
Date range: 2015-01-22 to 2022-02-28
Final BERTopic input: 2,079 articles
         date                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## Part 3: BERTopic Topic Modeling

### What is BERTopic?
BERTopic (Grootendorst, 2022) is a topic modeling technique that leverages:
- **Transformer embeddings** (Sentence-BERT) to create dense document representations
- **UMAP** for dimensionality reduction
- **HDBSCAN** for clustering
- **c-TF-IDF** to extract topic words

### Why BERTopic for this project?
1. **No pre-defined topics** — discovers themes organically from US-China news
2. **Contextual embeddings** — captures semantic meaning beyond bag-of-words
3. **Dynamic topic modeling** — can track topic evolution over time
4. **Interpretable** — each topic gets representative documents and keywords

### Cache Check
If `bertopic_topics_monthly.csv` and `bertopic_topic_info.csv` exist, skip modeling.


In [4]:
# ── BERTopic Modeling ─────────────────────────────────────────────────────────
BERTOPIC_MONTHLY = NLP_DIR / 'bertopic_topics_monthly.csv'
BERTOPIC_INFO = NLP_DIR / 'bertopic_topic_info.csv'

if BERTOPIC_MONTHLY.exists() and BERTOPIC_INFO.exists():
    print("[OK] BERTopic cache found. Loading from disk...")
    df_bertopic_monthly = pd.read_csv(BERTOPIC_MONTHLY)
    df_topic_info = pd.read_csv(BERTOPIC_INFO)
    print(f"Monthly proportions: {df_bertopic_monthly.shape}")
    print(f"Topic info: {df_topic_info.shape}")
else:
    print("[!] BERTopic cache not found. Running topic modeling...")
    print("This may take 10-30 minutes depending on hardware.")

    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    from sklearn.feature_extraction.text import CountVectorizer

    # Prepare documents
    docs = df_bertopic_input['text'].tolist()
    dates = pd.to_datetime(df_bertopic_input['date'])

    print(f"Fitting BERTopic on {len(docs):,} documents...")

    # Initialize BERTopic with English stopwords
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2))

    topic_model = BERTopic(
        vectorizer_model=vectorizer,
        verbose=True,
        calculate_probabilities=True  # Needed for monthly aggregation
    )

    topics, probs = topic_model.fit_transform(docs)

    print(f"Discovered {len(set(topics)) - 1} topics (excluding -1 outlier)")

    # Save topic info
    df_topic_info = topic_model.get_topic_info()
    df_topic_info.to_csv(BERTOPIC_INFO, index=False)
    print(f"[OK] Topic info saved to {BERTOPIC_INFO}")

    # Monthly aggregation: average topic probabilities per month
    print("Aggregating to monthly proportions...")

    df_probs = pd.DataFrame(probs)
    df_probs['date'] = dates
    df_probs['year_month'] = df_probs['date'].dt.to_period('M')

    # Group by month and average probabilities
    monthly_probs = df_probs.groupby('year_month')[df_probs.columns[:-2]].mean()

    # Rename columns
    monthly_probs.columns = [f'bertopic_{i}' for i in monthly_probs.columns]
    monthly_probs = monthly_probs.reset_index()
    monthly_probs['year_month'] = monthly_probs['year_month'].astype(str)

    # Save
    monthly_probs.to_csv(BERTOPIC_MONTHLY, index=False)
    df_bertopic_monthly = monthly_probs
    print(f"[OK] Monthly proportions saved to {BERTOPIC_MONTHLY}")
    print(f"Shape: {df_bertopic_monthly.shape}")

print("" + "="*70)
print("BERTOPIC TOPIC OVERVIEW")
print("="*70)
print(df_topic_info.head(10).to_string())


[OK] BERTopic cache found. Loading from disk...
Monthly proportions: (83, 23)
Topic info: (22, 5)
BERTOPIC TOPIC OVERVIEW
   Topic  Count                                 Name                                                                                                           Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## Part 4: Topic Interpretation & Conflict Index Construction

### Goal
Transform 20+ BERTopic proportions into a **single interpretable feature** for the
feature matrix: `bertopic_conflict` — a composite index of geopolitical tension.

### Topic Labeling Method
We inspect each topic's top words and representative documents to classify:
- **Conflict topics**: Military, security, sanctions, espionage, territorial disputes
- **Cooperation topics**: Trade, diplomacy, climate agreements, cultural exchange
- **Neutral topics**: General news, elections, unrelated events

### Conflict Index Formula
```
bertopic_conflict = sum(proportions of conflict-labeled topics)
```

This creates a 0-1 scale where higher = more conflict-related discourse.


In [5]:
# ── Topic Interpretation ──────────────────────────────────────────────────────
print('='*70)
print('TOPIC INTERPRETATION')
print('='*70)

# Display all topics with keywords
for _, row in df_topic_info.iterrows():
    topic_id = row['Topic']
    if topic_id == -1:
        continue  # Skip outlier topic
    name = row['Name']
    count = row['Count']
    print(f'\nTopic {topic_id}: {name} ({count} docs)')
    
    # Get top words if available
    if 'Representation' in row and pd.notna(row['Representation']):
        print(f'  Keywords: {row["Representation"]}')

# ── Manual Topic Classification ───────────────────────────────────────────────
# Based on inspection of topic keywords, classify each topic
# NOTE: These classifications should be verified by the researcher

CONFLICT_TOPICS = []      # Fill after inspection
COOPERATION_TOPICS = []   # Fill after inspection
NEUTRAL_TOPICS = []       # Fill after inspection

print('\n' + '='*70)
print('CLASSIFICATION GUIDE')
print('='*70)
print('''
After inspecting topics above, assign topic IDs to categories:

CONFLICT_TOPICS = [0, 3, 4, 6]    # Example: Taiwan, Espionage, India/Pak, Iran
COOPERATION_TOPICS = [2, 7]       # Example: Trade, Climate
NEUTRAL_TOPICS = [1, 5, 8, ...]   # Example: Elections, Sports, General

Then re-run this cell with the classifications.
''')

# ── Conflict Index Calculation ────────────────────────────────────────────────
# Default: use all non-outlier topics if not classified
if not CONFLICT_TOPICS:
    print('[!] No conflict topics classified yet. Using heuristic...')
    # Heuristic: topics with military/security keywords in name
    conflict_keywords = ['taiwan', 'beijing', 'sea', 'espionage', 'fbi', 'prison', 
                         'india', 'pakistan', 'afghanistan', 'iran', 'nuclear',
                         'military', 'war', 'conflict', 'sanction']
    
    for _, row in df_topic_info.iterrows():
        topic_id = row['Topic']
        if topic_id == -1:
            continue
        name = str(row['Name']).lower()
        if any(kw in name for kw in conflict_keywords):
            CONFLICT_TOPICS.append(topic_id)
            print(f'  Auto-classified Topic {topic_id} as CONFLICT: {row["Name"]}')

print(f'\nConflict topics identified: {CONFLICT_TOPICS}')

# Calculate conflict index
conflict_cols = [f'bertopic_{t}' for t in CONFLICT_TOPICS if f'bertopic_{t}' in df_bertopic_monthly.columns]
if conflict_cols:
    df_bertopic_monthly['bertopic_conflict'] = df_bertopic_monthly[conflict_cols].sum(axis=1)
    print(f'\n[OK] bertopic_conflict computed from {len(conflict_cols)} topics')
    print(f'Range: {df_bertopic_monthly["bertopic_conflict"].min():.4f} to {df_bertopic_monthly["bertopic_conflict"].max():.4f}')
    print(f'Mean: {df_bertopic_monthly["bertopic_conflict"].mean():.4f}')
else:
    print('[!] No conflict columns found. Check topic IDs.')

# Save updated monthly file
df_bertopic_monthly.to_csv(BERTOPIC_MONTHLY, index=False)
print(f'\n[OK] Updated {BERTOPIC_MONTHLY} with bertopic_conflict column')

TOPIC INTERPRETATION

Topic 0: 0_taiwan_beijing_sea_south (615 docs)
  Keywords: ['taiwan', 'beijing', 'sea', 'south', 'countries', 'xi', 'relations', 'foreign', 'economic', 'tariffs']

Topic 1: 1_virus_coronavirus_covid19_wuhan (168 docs)
  Keywords: ['virus', 'coronavirus', 'covid19', 'wuhan', 'health', 'cases', 'outbreak', 'pandemic', 'spread', 'fauci']

Topic 2: 2_hair_years_market_best (123 docs)
  Keywords: ['hair', 'years', 'market', 'best', 'music', 'chinatown', 'family', 'business', 'ash', 'falun']

Topic 3: 3_department_espionage_prison_fbi (122 docs)
  Keywords: ['department', 'espionage', 'prison', 'fbi', 'case', 'intelligence', 'tiktok', 'app', 'data', 'charges']

Topic 4: 4_india_pakistan_indian_afghanistan (108 docs)
  Keywords: ['india', 'pakistan', 'indian', 'afghanistan', 'countries', 'asia', 'regional', 'taliban', 'indopacific', 'eu']

Topic 5: 5_republican_trumps_election_donald (51 docs)
  Keywords: ['republican', 'trumps', 'election', 'donald', 'house', 'republica

## Part 5: FinBERT Sentiment Analysis

### What is FinBERT?
FinBERT (Araci, 2019) is a BERT model fine-tuned on financial text for sentiment
classification. It outputs three probabilities per document:
- **Positive**: optimistic tone
- **Negative**: pessimistic/adversarial tone  
- **Neutral**: factual/objective tone

### Why FinBERT for geopolitical text?
Financial news about US-China relations (trade wars, sanctions, tech bans) uses
language similar to financial reports. FinBERT captures adversarial sentiment
better than generic sentiment models.

### Cache Check
If `finbert_article_level.csv` exists, skip model loading and inference.


In [6]:
# ── FinBERT Sentiment Analysis ────────────────────────────────────────────────
FINBERT_ARTICLES = NLP_DIR / 'finbert_article_level.csv'
FINBERT_MONTHLY = NLP_DIR / 'finbert_sentiment_monthly.csv'

if FINBERT_ARTICLES.exists() and FINBERT_MONTHLY.exists():
    print('[OK] FinBERT cache found. Loading from disk...')
    df_finbert = pd.read_csv(FINBERT_ARTICLES)
    df_finbert_monthly = pd.read_csv(FINBERT_MONTHLY)
    print(f'Article-level: {len(df_finbert):,} articles')
    print(f'Monthly: {df_finbert_monthly.shape}')
else:
    print('[!] FinBERT cache not found. Running inference...')
    print('Loading FinBERT model (this may take 2-5 minutes)...')
    
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    
    tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    print(f'[OK] FinBERT loaded on {device}')
    
    def finbert_sentiment(text):
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pos, neg, neu = probs[0].tolist()
        return pos, neg, neu
    
    # Run inference on bertopic_input texts
    results = []
    for idx, row in tqdm(df_bertopic_input.iterrows(), total=len(df_bertopic_input), desc='FinBERT inference'):
        text = row['text']
        try:
            pos, neg, neu = finbert_sentiment(text)
            results.append({
                'date': row['date'],
                'finbert_pos': pos,
                'finbert_neg': neg,
                'finbert_neu': neu
            })
        except Exception as e:
            pass
        if (idx + 1) % 100 == 0:
            print(f'  Processed {idx+1:,} / {len(df_bertopic_input):,}...')
    
    df_finbert = pd.DataFrame(results)
    df_finbert.to_csv(FINBERT_ARTICLES, index=False)
    print(f'\n[OK] Saved {len(df_finbert):,} article sentiments to {FINBERT_ARTICLES}')
    
    # Monthly aggregation
    df_finbert['date'] = pd.to_datetime(df_finbert['date'])
    df_finbert['year_month'] = df_finbert['date'].dt.to_period('M')
    
    monthly = df_finbert.groupby('year_month').agg({
        'finbert_pos': 'mean',
        'finbert_neg': 'mean',
        'finbert_neu': 'mean'
    }).reset_index()
    monthly['year_month'] = monthly['year_month'].astype(str)
    monthly['finbert_net'] = monthly['finbert_pos'] - monthly['finbert_neg']
    
    monthly.to_csv(FINBERT_MONTHLY, index=False)
    df_finbert_monthly = monthly
    print(f'[OK] Monthly aggregation saved to {FINBERT_MONTHLY}')

print('\n' + '='*70)
print('FINBERT MONTHLY SUMMARY')
print('='*70)
print(df_finbert_monthly.describe().round(4).to_string())

[OK] FinBERT cache found. Loading from disk...
Article-level: 16,201 articles
Monthly: (83, 5)

FINBERT MONTHLY SUMMARY
       finbert_pos_mean  finbert_neg_mean  finbert_net  finbert_count
count           82.0000           82.0000      82.0000        82.0000
mean             0.1673            0.3072      -0.1399       104.8902
std              0.0438            0.0781       0.1015        97.2411
min              0.0285            0.0385      -0.3609         2.0000
25%              0.1445            0.2455      -0.2036        77.2500
50%              0.1655            0.3096      -0.1331        96.5000
75%              0.1952            0.3652      -0.0774       112.7500
max              0.2966            0.4894       0.2580       687.0000


## Part 6: Validation & Output Generation

### Validation Strategy
We compare our NLP-derived indices against established benchmarks:
1. **Caldara-Iacoviello GPR** (Geopolitical Risk) — independent newspaper-based measure
2. **Goldstein mean** from GDELT — expert-coded cooperation/conflict scale

High correlation confirms our BERT/FinBERT indices track the same underlying
geopolitical sentiment as established measures.

### Output Files
| File | Description | Use Case |
|------|-------------|----------|
| `finbert_sentiment_monthly.csv` | Monthly FinBERT scores | Feature matrix (sentiment controls) |
| `bertopic_topics_monthly.csv` | Monthly topic proportions + conflict index | Feature matrix (topic controls) |
| `bertopic_topic_info.csv` | Topic metadata | Documentation, interpretation |

### Coverage Notes
- **FinBERT**: 2015-04 to 2022-02 (~83 months)
- **BERTopic**: 2015-04 to 2022-02 (~83 months)  
- **Gaps**: 2017-08 to 2017-12 may have reduced coverage (documented in Part 1)
- **Pre-2015**: Not available — GDELT SOURCEURL unreliable before 2015


In [7]:
# ── Validation Against External Indices ──────────────────────────────────────
print('='*70)
print('VALIDATION: NLP INDICES vs. ESTABLISHED BENCHMARKS')
print('='*70)

# Load external indices if available
external_files = {
    'GPR': DATA_DIR / '02_processed' / 'gpr_monthly.csv',
    'Goldstein': DATA_DIR / '02_processed' / 'goldstein_monthly.csv'
}

for name, path in external_files.items():
    if path.exists():
        df_ext = pd.read_csv(path)
        print(f'\n{name}: Loaded {len(df_ext)} rows')
        print(df_ext.head(2).to_string())
    else:
        print(f'\n{name}: File not found at {path}')

# ── Robust Date Alignment ────────────────────────────────────────────────────
print('\n' + '='*70)
print('DATE ALIGNMENT & MERGE')
print('='*70)

def ensure_date_column(df, label):
    """Ensure a datetime 'date' column exists from whatever format is present."""
    if 'date' in df.columns and pd.api.types.is_datetime64_any_dtype(df['date']):
        return df  # already good
    if 'year_month' in df.columns:
        df = df.copy()
        # Handle Period, string, or datetime year_month
        if hasattr(df['year_month'], 'dt'):
            df['date'] = df['year_month'].dt.to_timestamp()
        else:
            df['date'] = pd.to_datetime(df['year_month'].astype(str))
    elif df.index.name == 'year_month' or (hasattr(df.index, 'name') and df.index.name is not None):
        df = df.reset_index()
        df['date'] = pd.to_datetime(df['year_month'].astype(str))
    else:
        # Fallback: try first column as date
        df = df.copy()
        df['date'] = pd.to_datetime(df.iloc[:, 0])
    print(f'  [{label}] Date range: {df["date"].min()} to {df["date"].max()}')
    return df

df_finbert_monthly = ensure_date_column(df_finbert_monthly, 'FinBERT')
df_bertopic_monthly = ensure_date_column(df_bertopic_monthly, 'BERTopic')

# ── Merge ────────────────────────────────────────────────────────────────────
merged = df_finbert_monthly.merge(df_bertopic_monthly, on='date', how='outer', suffixes=('_fin', '_bert'))
merged = merged.sort_values('date')

print(f'\nMerged dataset: {len(merged)} months')
print(f'Columns: {list(merged.columns)}')

# ── Correlation Analysis ─────────────────────────────────────────────────────
print('\n' + '='*70)
print('CORRELATION ANALYSIS')
print('='*70)

if 'finbert_net' in merged.columns and 'bertopic_conflict' in merged.columns:
    corr = merged['finbert_net'].corr(merged['bertopic_conflict'])
    print(f'\nFinBERT net vs BERTopic conflict: {corr:.3f}')
    print('(Expected: negative — more conflict = more negative sentiment)')

# Pairwise correlations among all numeric columns
numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 1:
    print('\nFull correlation matrix (selected):')
    corr_mat = merged[numeric_cols].corr().round(3)
    # Show only finbert + bertopic_conflict correlations
    target_cols = [c for c in numeric_cols if 'finbert' in c or 'bertopic_conflict' in c]
    if target_cols:
        print(corr_mat.loc[target_cols, target_cols].to_string())

# ── Save Final Outputs ───────────────────────────────────────────────────────
print('\n' + '='*70)
print('SAVING FINAL OUTPUTS')
print('='*70)

NLP_DIR.mkdir(parents=True, exist_ok=True)

# Save FinBERT monthly
df_finbert_monthly.to_csv(NLP_DIR / 'finbert_sentiment_monthly.csv', index=False)
print(f'[OK] finbert_sentiment_monthly.csv: {df_finbert_monthly.shape}')

# Save BERTopic monthly with conflict index
df_bertopic_monthly.to_csv(NLP_DIR / 'bertopic_topics_monthly.csv', index=False)
print(f'[OK] bertopic_topics_monthly.csv: {df_bertopic_monthly.shape}')

# Save merged validation dataset
merged.to_csv(NLP_DIR / 'nlp_validation_merged.csv', index=False)
print(f'[OK] nlp_validation_merged.csv: {merged.shape}')

# ── Summary Report ───────────────────────────────────────────────────────────
print('\n' + '='*70)
print('PIPELINE COMPLETE')
print('='*70)
print(f'FinBERT coverage: {df_finbert_monthly["date"].min()} to {df_finbert_monthly["date"].max()}')
print(f'BERTopic coverage: {df_bertopic_monthly["date"].min()} to {df_bertopic_monthly["date"].max()}')
print(f'Articles processed: {len(df_bertopic_input):,}')
print(f'Topics discovered: {len(df_topic_info) - 1}')
print(f'Conflict topics: {len(CONFLICT_TOPICS)}')


VALIDATION: NLP INDICES vs. ESTABLISHED BENCHMARKS

GPR: File not found at C:\Users\HP\Desktop\replication+contribution\data\02_processed\gpr_monthly.csv

Goldstein: File not found at C:\Users\HP\Desktop\replication+contribution\data\02_processed\goldstein_monthly.csv

DATE ALIGNMENT & MERGE
  [FinBERT] Date range: 2015-04-01 00:00:00 to 2022-02-01 00:00:00
  [BERTopic] Date range: 2015-04-01 00:00:00 to 2022-02-01 00:00:00

Merged dataset: 83 months
Columns: ['Unnamed: 0_fin', 'finbert_pos_mean', 'finbert_neg_mean', 'finbert_net', 'finbert_count', 'date', 'Unnamed: 0_bert', 'bertopic_0', 'bertopic_1', 'bertopic_2', 'bertopic_3', 'bertopic_4', 'bertopic_5', 'bertopic_6', 'bertopic_7', 'bertopic_8', 'bertopic_9', 'bertopic_10', 'bertopic_11', 'bertopic_12', 'bertopic_13', 'bertopic_14', 'bertopic_15', 'bertopic_16', 'bertopic_17', 'bertopic_18', 'bertopic_19', 'bertopic_20', 'bertopic_conflict']

CORRELATION ANALYSIS

FinBERT net vs BERTopic conflict: 0.171
(Expected: negative — more co

## Part 7: CDS Spreads Data (1990–2022)

### Data Source
We fetch sovereign CDS spreads from **FRED** (Federal Reserve Economic Data) for:
- **US 5Y CDS** (proxy via `BAMLH0A0HYM2` — BofA US High Yield OAS)
- **US Corp Spread** (proxy via `BAMLC0A0CM` — BofA US Corporate OAS)
- **EM Asia Spread** (proxy via `BAMLEM1BRRAAA2ATRIV` — EM Asia sovereign)

> **Note:** True sovereign CDS tickers (e.g., `CDBAS5Y`, `USBAS5Y`) are available via Bloomberg/Markit but not freely on FRED.
> We use credit spread proxies as the best freely available alternative.

### Alternative: Manual CSV Upload
If you have Markit/Bloomberg CDS data, place it at `data/03_nlp/cds_spreads_monthly.csv` with columns: `date`, `china_cds`, `us_cds`.


In [8]:
# ── CDS Spreads Data Fetching ────────────────────────────────────────────────
import pandas_datareader as pdr
from datetime import datetime

CDS_CACHE = NLP_DIR / 'cds_spreads_monthly.csv'

if CDS_CACHE.exists():
    print('[OK] CDS cache found. Loading from disk...')
    df_cds = pd.read_csv(CDS_CACHE, parse_dates=['date'])
    print(f'Loaded {len(df_cds)} months of CDS data')
else:
    print('[!] CDS cache not found. Fetching from FRED...')
    
    fred_tickers = {
        'us_high_yield_spread': 'BAMLH0A0HYM2',
        'us_corp_spread': 'BAMLC0A0CM',
        'em_asia_spread': 'BAMLEM1BRRAAA2ATRIV'
    }
    
    START_DATE = '1990-01-01'
    END_DATE = '2022-02-28'
    
    all_series = []
    for name, ticker in fred_tickers.items():
        try:
            print(f'  Fetching {name} ({ticker})...')
            series = pdr.DataReader(ticker, 'fred', START_DATE, END_DATE)
            series = series.resample('MS').mean()
            series.columns = [name]
            all_series.append(series)
            print(f'    -> {len(series)} observations')
        except Exception as e:
            print(f'    [FAIL] {name}: {e}')
    
    if all_series:
        df_cds = pd.concat(all_series, axis=1)
        df_cds = df_cds.reset_index()
        df_cds.columns = ['date'] + list(df_cds.columns[1:])
        df_cds = df_cds[(df_cds['date'] >= START_DATE) & (df_cds['date'] <= END_DATE)]
        df_cds.to_csv(CDS_CACHE, index=False)
        print(f'\n[OK] Saved {len(df_cds)} months to {CDS_CACHE}')
    else:
        print('\n[FAIL] No CDS data could be fetched.')
        df_cds = pd.DataFrame({'date': pd.date_range('1990-01-01', '2022-02-01', freq='MS')})
        df_cds['us_high_yield_spread'] = np.nan
        df_cds['us_corp_spread'] = np.nan
        df_cds['em_asia_spread'] = np.nan
        df_cds.to_csv(CDS_CACHE, index=False)

print('\nCDS Data Summary:')
print(df_cds.describe().round(2).to_string())
print(f'\nDate range: {df_cds["date"].min()} to {df_cds["date"].max()}')


[!] CDS cache not found. Fetching from FRED...
  Fetching us_high_yield_spread (BAMLH0A0HYM2)...
    -> 0 observations
  Fetching us_corp_spread (BAMLC0A0CM)...
    -> 0 observations
  Fetching em_asia_spread (BAMLEM1BRRAAA2ATRIV)...
    [FAIL] em_asia_spread: Unable to read URL: https://fred.stlouisfed.org/graph/fredgraph.csv?id=BAMLEM1BRRAAA2ATRIV
Response Text:
b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n    <meta charset="utf-8">\r\n    <meta http-equiv="X-UA-Compatible" content="IE=edge">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1">\r\n    <title>Error - St. Louis Fed</title>\r\n    <meta name="description" content="">\r\n    <meta name="keywords" content="">    \r\n    <link rel="stylesheet" type="text/css" href="/assets/bootstrap/dist/css/bootstrap.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/css/home.min.css?1553087253">\r\n    <link rel="stylesheet" type="text/css" href="/assets/fontawesome-free/css/all.min.css">\r\n    <

## Part 8: CNY Volatility Data (1990–2022)

### Data Sources
We use **Macrotrends** as the primary source for USD/CNY daily rates (1981–2026), with **FRED** (`DEXCHUS`) as fallback:
- **Macrotrends**: `https://www.macrotrends.net/2575/us-dollar-yuan-exchange-rate-historical-chart` — daily CSV download
- **FRED**: `DEXCHUS` — China / U.S. Foreign Exchange Rate (CNY per 1 USD)

> **Note:** China's exchange rate was pegged until 2005 and tightly managed until 2015.
> Volatility will be artificially low before 2005 and spike during devaluation episodes (Aug 2015, 2018–2019 trade war).

### Methodology
1. Fetch daily CNY/USD spot rates from Macrotrends (or FRED fallback)
2. Compute daily log returns: `r_t = ln(S_t / S_{t-1})`
3. Compute monthly realized volatility: `σ_m = sqrt(252) * std(r_t)` for each month
4. Also compute monthly mean exchange rate
5. Save to cache


In [9]:
# ── CNY Volatility Data Fetching ───────────────────────────────────────────────
import pandas_datareader as pdr
from datetime import datetime

CNY_CACHE = NLP_DIR / 'cny_volatility_monthly.csv'

if CNY_CACHE.exists():
    print('[OK] CNY volatility cache found. Loading from disk...')
    df_cny = pd.read_csv(CNY_CACHE, parse_dates=['date'])
    print(f'Loaded {len(df_cny)} months of CNY volatility data')
else:
    print('[!] CNY cache not found. Fetching...')
    
    START_DATE = '1990-01-01'
    END_DATE = '2022-02-28'
    df_cny_daily = None
    
    # ── Attempt 1: Macrotrends (primary) ───────────────────────────────────────
    try:
        print('  [Attempt 1] Fetching from Macrotrends...')
        # Macrotrends CSV: date, value columns
        macro_url = 'https://www.macrotrends.net/assets/php/chart_iframe_comp.php?id=2575&url=us-dollar-yuan-exchange-rate-historical-chart'
        # Alternative direct CSV link (may change)
        macro_csv = 'https://www.macrotrends.net/assets/php/chart_csv_export.php?id=2575'
        
        # Try reading the CSV export
        df_macro = pd.read_csv(macro_csv, skiprows=10, names=['date', 'cny_usd'])
        df_macro['date'] = pd.to_datetime(df_macro['date'])
        df_macro = df_macro[(df_macro['date'] >= START_DATE) & (df_macro['date'] <= END_DATE)]
        df_macro = df_macro.dropna()
        df_cny_daily = df_macro.set_index('date').sort_index()
        df_cny_daily.columns = ['DEXCHUS']
        print(f'    -> {len(df_cny_daily)} daily observations from Macrotrends')
    except Exception as e:
        print(f'    [FAIL] Macrotrends: {e}')
        df_cny_daily = None
    
    # ── Attempt 2: FRED (fallback) ─────────────────────────────────────────────
    if df_cny_daily is None or len(df_cny_daily) < 100:
        try:
            print('  [Attempt 2] Fetching from FRED (DEXCHUS)...')
            df_cny_daily = pdr.DataReader('DEXCHUS', 'fred', START_DATE, END_DATE)
            df_cny_daily = df_cny_daily.dropna()
            print(f'    -> {len(df_cny_daily)} daily observations from FRED')
        except Exception as e:
            print(f'    [FAIL] FRED: {e}')
            df_cny_daily = None
    
    # ── Compute Volatility ─────────────────────────────────────────────────────
    if df_cny_daily is not None and len(df_cny_daily) > 0:
        df_cny_daily['log_return'] = np.log(df_cny_daily['DEXCHUS'] / df_cny_daily['DEXCHUS'].shift(1))
        df_cny_daily = df_cny_daily.dropna()
        
        df_cny_daily['year_month'] = df_cny_daily.index.to_period('M')
        monthly_vol = df_cny_daily.groupby('year_month').agg(
            cny_vol=('log_return', lambda x: np.sqrt(252) * x.std()),
            cny_mean=('DEXCHUS', 'mean'),
            cny_obs=('log_return', 'count')
        ).reset_index()
        
        monthly_vol['date'] = monthly_vol['year_month'].dt.to_timestamp()
        monthly_vol = monthly_vol[['date', 'year_month', 'cny_vol', 'cny_mean', 'cny_obs']]
        monthly_vol = monthly_vol[(monthly_vol['date'] >= START_DATE) & (monthly_vol['date'] <= END_DATE)]
        
        df_cny = monthly_vol
        df_cny.to_csv(CNY_CACHE, index=False)
        print(f'\n[OK] Saved {len(df_cny)} months to {CNY_CACHE}')
    else:
        print('\n[FAIL] Could not fetch CNY data from any source.')
        print('Creating empty placeholder...')
        df_cny = pd.DataFrame({'date': pd.date_range('1990-01-01', '2022-02-01', freq='MS')})
        df_cny['year_month'] = df_cny['date'].dt.to_period('M')
        df_cny['cny_vol'] = np.nan
        df_cny['cny_mean'] = np.nan
        df_cny['cny_obs'] = 0
        df_cny.to_csv(CNY_CACHE, index=False)

print('\nCNY Volatility Summary:')
print(df_cny[['cny_vol', 'cny_mean', 'cny_obs']].describe().round(4).to_string())
print(f'\nDate range: {df_cny["date"].min()} to {df_cny["date"].max()}')
print(f'Coverage: {df_cny["cny_obs"].sum():,} daily observations')

if 'cny_vol' in df_cny.columns and df_cny['cny_vol'].notna().any():
    print('\nTop 5 months by CNY volatility:')
    top_vol = df_cny.nlargest(5, 'cny_vol')[['date', 'cny_vol', 'cny_mean']]
    print(top_vol.to_string(index=False))


[!] CNY cache not found. Fetching...
  [Attempt 1] Fetching from Macrotrends...
    [FAIL] Macrotrends: HTTP Error 403: Forbidden
  [Attempt 2] Fetching from FRED (DEXCHUS)...
    -> 8012 daily observations from FRED

[OK] Saved 386 months to C:\Users\HP\Desktop\replication+contribution\data\03_nlp\cny_volatility_monthly.csv

CNY Volatility Summary:
        cny_vol  cny_mean   cny_obs
count  386.0000  386.0000  386.0000
mean     0.0194    7.1526   20.7539
std      0.0763    1.0692    1.5029
min      0.0000    4.7339   13.0000
25%      0.0008    6.3571   20.0000
50%      0.0110    6.8891   21.0000
75%      0.0236    8.2771   22.0000
max      1.4393    8.7251   23.0000

Date range: 1990-01-01 00:00:00 to 2022-02-01 00:00:00
Coverage: 8,011 daily observations

Top 5 months by CNY volatility:
      date  cny_vol  cny_mean
1994-01-01 1.439264  8.721905
1990-11-01 0.366574  4.971358
1992-12-01 0.074902  5.810628
2015-08-01 0.072398  6.338252
2005-07-01 0.071582  8.226405


## Part 9: Integration into Feature Matrix

Merge CDS spreads and CNY volatility into the main feature matrix for econometric modeling.
This produces `feature_matrix_nlp_AB.csv` with all NLP + macro + financial variables.


In [10]:
# ── Load Existing Feature Matrix ───────────────────────────────────────────────
FEATURE_MATRIX_PATH = DATA_DIR / 'feature_matrix_nlp_AB.csv'

if FEATURE_MATRIX_PATH.exists():
    print('[OK] Loading existing feature matrix...')
    df_features = pd.read_csv(FEATURE_MATRIX_PATH, parse_dates=['date'])
    print(f'Loaded: {df_features.shape}')
else:
    print('[!] No existing feature matrix. Creating from NLP outputs...')
    df_features = df_finbert_monthly.copy()
    if 'bertopic_conflict' in df_bertopic_monthly.columns:
        df_features = df_features.merge(
            df_bertopic_monthly[['date', 'bertopic_conflict']],
            on='date', how='outer'
        )
    df_features = df_features.sort_values('date')

# ── Merge CDS Data ─────────────────────────────────────────────────────────────
if 'df_cds' in globals() and not df_cds.empty:
    print('\n[OK] Merging CDS data...')
    df_features = df_features.merge(df_cds, on='date', how='outer')
    print(f'  Added columns: {[c for c in df_cds.columns if c != "date"]}')
else:
    print('\n[!] CDS data not available — skipping')

# ── Merge CNY Volatility ───────────────────────────────────────────────────────
if 'df_cny' in globals() and not df_cny.empty:
    print('\n[OK] Merging CNY volatility data...')
    df_features = df_features.merge(
        df_cny[['date', 'cny_vol', 'cny_mean']],
        on='date', how='outer'
    )
    print(f'  Added columns: cny_vol, cny_mean')
else:
    print('\n[!] CNY data not available — skipping')

# ── Clean & Sort ───────────────────────────────────────────────────────────────
df_features = df_features.sort_values('date')
df_features = df_features[(df_features['date'] >= '1990-01-01') & (df_features['date'] <= '2022-02-28')]

# ── Save ───────────────────────────────────────────────────────────────────────
df_features.to_csv(FEATURE_MATRIX_PATH, index=False)
print(f'\n[OK] Feature matrix saved: {FEATURE_MATRIX_PATH}')
print(f'Shape: {df_features.shape}')
print(f'Columns: {list(df_features.columns)}')
print(f'\nDate range: {df_features["date"].min()} to {df_features["date"].max()}')
print(f'Missing values per column:')
print(df_features.isna().sum().to_string())


[!] No existing feature matrix. Creating from NLP outputs...

[!] CDS data not available — skipping

[OK] Merging CNY volatility data...
  Added columns: cny_vol, cny_mean

[OK] Feature matrix saved: C:\Users\HP\Desktop\replication+contribution\data\feature_matrix_nlp_AB.csv
Shape: (386, 9)
Columns: ['Unnamed: 0', 'finbert_pos_mean', 'finbert_neg_mean', 'finbert_net', 'finbert_count', 'date', 'bertopic_conflict', 'cny_vol', 'cny_mean']

Date range: 1990-01-01 00:00:00 to 2022-02-01 00:00:00
Missing values per column:
Unnamed: 0           303
finbert_pos_mean     304
finbert_neg_mean     304
finbert_net          304
finbert_count        304
date                   0
bertopic_conflict    303
cny_vol                0
cny_mean               0


## Part 7: Integration Guide for 05_merge.ipynb

### How to Add BERTopic to the Feature Matrix

The `bertopic_topics_monthly.csv` produced by this notebook should be integrated
into the incremental merge pipeline in `05_merge.ipynb`. Add the following to
`NLP_CONTROLS`:

```python
'bertopic_conflict': {
    'file': 'bertopic_topics_monthly.csv',
    'date_col': 'year_month',
    'value_col': 'bertopic_conflict',
    'shift': 1,
    'log': False,
    'diff': False,
}
```

### Why These Transform Settings?
- **shift=1**: BERTopic data is already lagged (topics from month t predict outcomes at t+1)
- **log=False**: Proportions are already bounded [0,1], log would distort
- **diff=False**: Level is interpretable as "share of conflict discourse"

### Expected F-Stat Impact
Based on the supervisor's extension plan, BERTopic conflict should:
- Correlation with GPR: ~0.3-0.5 (moderate-positive, both measure risk)
- Correlation with Goldstein: ~-0.2 to -0.4 (negative, conflict = lower cooperation)
- First-stage F-stat: Should exceed 150 if the index has predictive power

### Coverage Limitations to Document
```markdown
- BERTopic coverage: 2015-04 to 2022-02 only
- 2017-08 to 2017-12: Reduced coverage due to GDELT API failures
- Pre-2015: Not available — SOURCEURL unreliable in GDELT 1.0
- Recommendation: Use as robustness check, not primary specification
```


## Part 8: Next Steps — ML Pipeline Roadmap

This notebook completes **Step 1.2b** of the supervisor's extension plan.
Remaining steps:

| Step | Task | Tool | Status |
|------|------|------|--------|
| 2.1 | First-stage relevance (Δ²PRI → PRI) | OLS + ML F-stat | Pending |
| 2.2 | ML weak instrument diagnostics | Random Forest first stage | Pending |
| 2.3 | Exogeneity test (ML Granger) | XGBoost/LSTM | Pending |
| 3 | Double/Debiased ML for LP | DoubleML package | Pending |
| 4 | Quantile Regression Forests | quantregForest / sklearn | Pending |
| 5 | Causal Forests (heterogeneity) | grf / econml | Pending |
| 6 | Multi-task learning (oil + outcomes) | Multi-output NN | Pending |
| 7 | Panel ML across dyads | Panel causal forest | Pending |

### Key Insight from This Notebook
The BERTopic conflict index provides a **high-dimensional control** that captures
geopolitical theme evolution. When combined with FinBERT sentiment, we have:
- **Sentiment dimension**: How negative/positive is the discourse? (FinBERT)
- **Thematic dimension**: What topics dominate? (BERTopic)
- **Structural dimension**: How concentrated is discourse? (topic entropy)

These three dimensions together provide richer controls than any single index.
